In [1]:
"""
Write DICOMs from raw (non-CNN) velocity data to validate the DICOM construction pipeline.

Produces two DICOM sets per patient:
  1. uncorrected/  — original uncorrected velocities + magnitude
  2. corrected/    — ground-truth corrected velocities + magnitude

Compare these in the PACS against the original DICOMs to verify
that the pipeline preserves velocity values and signs correctly.
"""
from pathlib import Path
from pydicom.uid import generate_uid

from vascular_superenhancement.utils.path_config import load_path_config
from vascular_superenhancement.data_management.patients import Patient
from vascular_superenhancement.data_management.nifti_to_dicom import NiftiToDicomConverter

# ── Configuration ──────────────────────────────────────────────────────────
PATIENT_IDS = ["Balboloop", "Dithigog", "Sepigoo"]
TIMEPOINTS  = None                      # None = all timepoints
OUTPUT_ROOT = Path("/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation")

pc = load_path_config("all_patients")
print(f"Output root: {OUTPUT_ROOT.resolve()}")

Found project root at: /home/ayeluru/vascular-superenhancement-4d-flow
Output root: /data/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation


In [2]:
for pid in PATIENT_IDS:
    print(f"\n{'='*60}")
    print(f"Patient: {pid}")
    print(f"{'='*60}")

    patient = Patient(path_config=pc, phonetic_id=pid, debug=False, config="all_patients")
    converter = NiftiToDicomConverter.from_patient(patient)
    num_tp = patient.num_timepoints

    # Paths to per-timepoint NIfTIs at corrected-velocity FOV
    mag_dir   = patient.flow_mag_per_timepoint_corr_fov_dir
    uncorr_vx = patient.flow_vx_per_timepoint_corr_fov_dir
    uncorr_vy = patient.flow_vy_per_timepoint_corr_fov_dir
    uncorr_vz = patient.flow_vz_per_timepoint_corr_fov_dir
    corr_vx   = patient.flow_vx_corr_per_timepoint_dir
    corr_vy   = patient.flow_vy_corr_per_timepoint_dir
    corr_vz   = patient.flow_vz_corr_per_timepoint_dir

    timepoints = TIMEPOINTS if TIMEPOINTS is not None else list(range(num_tp))
    print(f"  Writing {len(timepoints)} timepoints …")

    # Shared UIDs across all timepoints (one study+series per set)
    study_uid_uncorr = generate_uid()
    series_uids_uncorr = {2: generate_uid(), 3: generate_uid(),
                          4: generate_uid(), 5: generate_uid()}
    study_uid_corr = generate_uid()
    series_uids_corr = {2: generate_uid(), 3: generate_uid(),
                        4: generate_uid(), 5: generate_uid()}

    for t in timepoints:
        mag_path = mag_dir / f"4d_flow_mag_{pid}_frame_{t:02d}.nii.gz"
        assert mag_path.exists(), f"Missing mag: {mag_path}"

        # ── 1. Uncorrected DICOMs ────────────────────────────────────
        uncorr_vel_paths = {
            "vx": uncorr_vx / f"4d_flow_vx_{pid}_frame_{t:02d}.nii.gz",
            "vy": uncorr_vy / f"4d_flow_vy_{pid}_frame_{t:02d}.nii.gz",
            "vz": uncorr_vz / f"4d_flow_vz_{pid}_frame_{t:02d}.nii.gz",
        }
        for k, p in uncorr_vel_paths.items():
            assert p.exists(), f"Missing uncorrected {k}: {p}"

        uncorr_out = OUTPUT_ROOT / pid / "uncorrected"

        print(f"  t={t}: writing uncorrected DICOMs → {uncorr_out}")
        converter.write_timepoint_with_velocities_to_dicoms(
            mag_prediction_path=mag_path,
            velocity_paths=uncorr_vel_paths,
            output_dir=uncorr_out,
            timepoint=t,
            study_uid=study_uid_uncorr,
            series_uids=series_uids_uncorr,
            overwrite=True,
            series_descriptions={
                2: "Validation Mag (raw)",
                3: "Validation Vx (uncorrected)",
                4: "Validation Vy (uncorrected)",
                5: "Validation Vz (uncorrected)",
            },
        )

        # ── 2. Corrected DICOMs ──────────────────────────────────────
        corr_vel_paths = {
            "vx": corr_vx / f"4d_flow_vx_corr_{pid}_frame_{t:02d}.nii.gz",
            "vy": corr_vy / f"4d_flow_vy_corr_{pid}_frame_{t:02d}.nii.gz",
            "vz": corr_vz / f"4d_flow_vz_corr_{pid}_frame_{t:02d}.nii.gz",
        }
        for k, p in corr_vel_paths.items():
            assert p.exists(), f"Missing corrected {k}: {p}"

        corr_out = OUTPUT_ROOT / pid / "corrected"

        print(f"  t={t}: writing corrected DICOMs   → {corr_out}")
        converter.write_timepoint_with_velocities_to_dicoms(
            mag_prediction_path=mag_path,
            velocity_paths=corr_vel_paths,
            output_dir=corr_out,
            timepoint=t,
            study_uid=study_uid_corr,
            series_uids=series_uids_corr,
            overwrite=True,
            series_descriptions={
                2: "Validation Mag (raw)",
                3: "Validation Vx (corrected GT)",
                4: "Validation Vy (corrected GT)",
                5: "Validation Vz (corrected GT)",
            },
        )

    print(f"  Done for {pid}")

print(f"\nAll done. DICOMs in: {OUTPUT_ROOT.resolve()}")


Patient: Balboloop


2026-03-10 21:51:30,620 - INFO - Successfully loaded existing 4D Flow catalog for patient Balboloop


  Writing 20 timepoints …
  t=0: writing uncorrected DICOMs → /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation/Balboloop/uncorrected


2026-03-10 21:51:36,518 - INFO - Using fixed DICOM intensity range: 0.0, 3473.0
2026-03-10 21:51:36,536 - INFO - Prediction intensity range: 0.0, 3470.0
2026-03-10 21:51:57,373 - INFO - Processed timepoint 0: 140 slices (mag + velocity)


  t=0: writing corrected DICOMs   → /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation/Balboloop/corrected


...

2026-03-10 22:32:51,263 - INFO - Using fixed DICOM intensity range: 0.0, 7001.0
2026-03-10 22:32:51,287 - INFO - Prediction intensity range: 0.0, 6998.0
2026-03-10 22:33:07,833 - INFO - Processed timepoint 19: 170 slices (mag + velocity)


  t=19: writing corrected DICOMs   → /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation/Sepigoo/corrected


2026-03-10 22:33:11,590 - INFO - Using fixed DICOM intensity range: 0.0, 7001.0
2026-03-10 22:33:11,613 - INFO - Prediction intensity range: 0.0, 6998.0


  Done for Sepigoo

All done. DICOMs in: /data/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation


2026-03-10 22:33:28,325 - INFO - Processed timepoint 19: 170 slices (mag + velocity)


In [3]:
"""
Negative control: write DICOMs for Balboloop with vz NEGATED (vx/vy unchanged).
If the PACS shows flipped vz directions, it proves the PACS isn't silently fixing signs.
"""
import nibabel as nib
import numpy as np
import SimpleITK as sitk
from pathlib import Path

pid = "Balboloop"
patient = Patient(path_config=pc, phonetic_id=pid, debug=False, config="all_patients")
converter = NiftiToDicomConverter.from_patient(patient)
num_tp = patient.num_timepoints

uncorr_vx_dir = patient.flow_vx_per_timepoint_corr_fov_dir
uncorr_vy_dir = patient.flow_vy_per_timepoint_corr_fov_dir
uncorr_vz_dir = patient.flow_vz_per_timepoint_corr_fov_dir
mag_dir = patient.flow_mag_per_timepoint_corr_fov_dir

# Temporary directory for negated NIfTIs
negated_dir = OUTPUT_ROOT / pid / "negated_niftis"
negated_dir.mkdir(parents=True, exist_ok=True)

negated_out = OUTPUT_ROOT / pid / "negated_vz_only"

study_uid = generate_uid()
series_uids = {2: generate_uid(), 3: generate_uid(),
               4: generate_uid(), 5: generate_uid()}

for t in range(num_tp):
    mag_path = mag_dir / f"4d_flow_mag_{pid}_frame_{t:02d}.nii.gz"

    # Negate each velocity component and save as temporary NIfTI
    neg_vel_paths = {}
    for comp, src_dir in [("vx", uncorr_vx_dir), ("vy", uncorr_vy_dir), ("vz", uncorr_vz_dir)]:
        src_path = src_dir / f"4d_flow_{comp}_{pid}_frame_{t:02d}.nii.gz"
        if comp == "vz":
            img = sitk.ReadImage(str(src_path))
            arr = sitk.GetArrayFromImage(img)
            neg_arr = -arr  # NEGATE vz only
            neg_img = sitk.GetImageFromArray(neg_arr)
            neg_img.CopyInformation(img)
            neg_path = negated_dir / f"neg_{comp}_frame_{t:02d}.nii.gz"
            sitk.WriteImage(neg_img, str(neg_path))
            neg_vel_paths[comp] = neg_path
        else:
            neg_vel_paths[comp] = src_path  # vx/vy unchanged

    print(f"  t={t}: writing negated DICOMs")
    converter.write_timepoint_with_velocities_to_dicoms(
        mag_prediction_path=mag_path,
        velocity_paths=neg_vel_paths,
        output_dir=negated_out,
        timepoint=t,
        study_uid=study_uid,
        series_uids=series_uids,
        overwrite=True,
        series_descriptions={
            2: "Validation Mag (raw)",
            3: "Vx (unchanged)",
            4: "Vy (unchanged)",
            5: "NEGATED Vz (negative control)",
        },
    )

print(f"\nDone. Negated DICOMs in: {negated_out}")

2026-03-10 23:20:10,013 - INFO - Successfully loaded existing 4D Flow catalog for patient Balboloop


  t=0: writing negated DICOMs


2026-03-10 23:20:15,458 - INFO - Using fixed DICOM intensity range: 0.0, 3473.0
2026-03-10 23:20:15,478 - INFO - Prediction intensity range: 0.0, 3470.0
2026-03-10 23:20:33,803 - INFO - Processed timepoint 0: 140 slices (mag + velocity)


  t=1: writing negated DICOMs


2026-03-10 23:20:37,977 - INFO - Using fixed DICOM intensity range: 0.0, 3476.0
2026-03-10 23:20:37,999 - INFO - Prediction intensity range: 0.0, 3472.0
2026-03-10 23:20:52,390 - INFO - Processed timepoint 1: 140 slices (mag + velocity)


...

2026-03-10 23:25:18,231 - INFO - Using fixed DICOM intensity range: 0.0, 3475.0
2026-03-10 23:25:18,251 - INFO - Prediction intensity range: 0.0, 3472.0
2026-03-10 23:25:30,958 - INFO - Processed timepoint 18: 140 slices (mag + velocity)


  t=19: writing negated DICOMs


2026-03-10 23:25:34,842 - INFO - Using fixed DICOM intensity range: 0.0, 3471.0
2026-03-10 23:25:34,864 - INFO - Prediction intensity range: 0.0, 3468.961000001058



Done. Negated DICOMs in: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation/Balboloop/negated_vz_only


2026-03-10 23:25:47,505 - INFO - Processed timepoint 19: 140 slices (mag + velocity)


In [4]:
"""
Negative control #2: write DICOMs for Balboloop with ALL velocity components negated.
"""
pid = "Balboloop"
patient = Patient(path_config=pc, phonetic_id=pid, debug=False, config="all_patients")
converter = NiftiToDicomConverter.from_patient(patient)
num_tp = patient.num_timepoints

uncorr_vx_dir = patient.flow_vx_per_timepoint_corr_fov_dir
uncorr_vy_dir = patient.flow_vy_per_timepoint_corr_fov_dir
uncorr_vz_dir = patient.flow_vz_per_timepoint_corr_fov_dir
mag_dir = patient.flow_mag_per_timepoint_corr_fov_dir

negated_dir = OUTPUT_ROOT / pid / "negated_all_niftis"
negated_dir.mkdir(parents=True, exist_ok=True)

negated_out = OUTPUT_ROOT / pid / "negated_all_vel"

study_uid = generate_uid()
series_uids = {2: generate_uid(), 3: generate_uid(),
               4: generate_uid(), 5: generate_uid()}

for t in range(num_tp):
    mag_path = mag_dir / f"4d_flow_mag_{pid}_frame_{t:02d}.nii.gz"

    neg_vel_paths = {}
    for comp, src_dir in [("vx", uncorr_vx_dir), ("vy", uncorr_vy_dir), ("vz", uncorr_vz_dir)]:
        src_path = src_dir / f"4d_flow_{comp}_{pid}_frame_{t:02d}.nii.gz"
        img = sitk.ReadImage(str(src_path))
        arr = sitk.GetArrayFromImage(img)
        neg_arr = -arr  # NEGATE all components
        neg_img = sitk.GetImageFromArray(neg_arr)
        neg_img.CopyInformation(img)
        neg_path = negated_dir / f"neg_{comp}_frame_{t:02d}.nii.gz"
        sitk.WriteImage(neg_img, str(neg_path))
        neg_vel_paths[comp] = neg_path

    print(f"  t={t}: writing negated-all DICOMs")
    converter.write_timepoint_with_velocities_to_dicoms(
        mag_prediction_path=mag_path,
        velocity_paths=neg_vel_paths,
        output_dir=negated_out,
        timepoint=t,
        study_uid=study_uid,
        series_uids=series_uids,
        overwrite=True,
        series_descriptions={
            2: "Validation Mag (raw)",
            3: "NEGATED Vx (negative control)",
            4: "NEGATED Vy (negative control)",
            5: "NEGATED Vz (negative control)",
        },
    )

print(f"\nDone. Negated-all DICOMs in: {negated_out}")

2026-03-10 23:44:20,714 - INFO - Successfully loaded existing 4D Flow catalog for patient Balboloop


  t=0: writing negated-all DICOMs


2026-03-10 23:44:25,573 - INFO - Using fixed DICOM intensity range: 0.0, 3473.0
2026-03-10 23:44:25,594 - INFO - Prediction intensity range: 0.0, 3470.0
2026-03-10 23:44:38,720 - INFO - Processed timepoint 0: 140 slices (mag + velocity)


  t=1: writing negated-all DICOMs


2026-03-10 23:44:43,285 - INFO - Using fixed DICOM intensity range: 0.0, 3476.0
2026-03-10 23:44:43,308 - INFO - Prediction intensity range: 0.0, 3472.0
2026-03-10 23:44:55,797 - INFO - Processed timepoint 1: 140 slices (mag + velocity)


...

2026-03-10 23:49:24,467 - INFO - Using fixed DICOM intensity range: 0.0, 3475.0
2026-03-10 23:49:24,487 - INFO - Prediction intensity range: 0.0, 3472.0
2026-03-10 23:49:36,941 - INFO - Processed timepoint 18: 140 slices (mag + velocity)


  t=19: writing negated-all DICOMs


2026-03-10 23:49:41,550 - INFO - Using fixed DICOM intensity range: 0.0, 3471.0
2026-03-10 23:49:41,571 - INFO - Prediction intensity range: 0.0, 3468.961000001058



Done. Negated-all DICOMs in: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/sample_patients/sandbox/raw_dicom_validation/Balboloop/negated_all_vel


2026-03-10 23:49:52,486 - INFO - Processed timepoint 19: 140 slices (mag + velocity)
